# Day 09. Exercise 00
# Regularization

## 0. Imports

In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import joblib


## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [31]:
df = pd.read_csv('../../src/data/dayofweek.csv')


In [32]:
X = df.drop(columns=['dayofweek'])
y = df['dayofweek']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)


In [33]:
def crossval(model, X, y, n_splits=10):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=False)
    accs = []
    for train_idx, valid_idx in skf.split(X, y):
        X_tr, X_v = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_v = y.iloc[train_idx], y.iloc[valid_idx]
        clone = model.__class__(**model.get_params())
        clone.fit(X_tr, y_tr)
        train_acc = accuracy_score(y_tr, clone.predict(X_tr))
        valid_acc = accuracy_score(y_v, clone.predict(X_v))
        accs.append(valid_acc)
        print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')
    print(f'Average accuracy on crossval is {np.mean(accs):.5f}')
    print(f'Std is {np.std(accs):.5f}')


## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [34]:
%%time
lr = LogisticRegression(random_state=21, fit_intercept=False, solver="saga")
crossval(lr, X_train, y_train)


train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64221   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
CPU times: user 289 ms, sys: 4.37 ms, total: 293 ms
Wall time: 295 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [35]:
lr_none = LogisticRegression(random_state=21, fit_intercept=False, penalty=None)
crossval(lr_none, X_train, y_train)


train -  0.66529   |   valid -  0.62963
train -  0.65705   |   valid -  0.65926
train -  0.66447   |   valid -  0.57778
train -  0.66529   |   valid -  0.62963
train -  0.66694   |   valid -  0.62222
train -  0.65952   |   valid -  0.57778
train -  0.65045   |   valid -  0.69630
train -  0.68673   |   valid -  0.61481
train -  0.66474   |   valid -  0.62687
train -  0.65651   |   valid -  0.61940
Average accuracy on crossval is 0.62537
Std is 0.03302


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

In [36]:
lr_l1 = LogisticRegression(random_state=21, fit_intercept=False, penalty='l1', solver='saga')
crossval(lr_l1, X_train, y_train)


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/

train -  0.63726   |   valid -  0.58519
train -  0.64056   |   valid -  0.61481
train -  0.62984   |   valid -  0.55556


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/

train -  0.64468   |   valid -  0.60000
train -  0.63397   |   valid -  0.57778
train -  0.63644   |   valid -  0.57778


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/

train -  0.63644   |   valid -  0.65926
train -  0.65622   |   valid -  0.57778
train -  0.64580   |   valid -  0.58209
train -  0.63756   |   valid -  0.62687
Average accuracy on crossval is 0.59571
Std is 0.02875


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [37]:
lr_l2 = LogisticRegression(random_state=21, fit_intercept=False, penalty='l2', solver='saga')
crossval(lr_l2, X_train, y_train)


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/leyla_iz/

train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64221   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/leyla_iz/

## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [38]:
%%time
svm = SVC(probability=True, kernel='linear', random_state=21)
crossval(svm, X_train, y_train)


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.70486   |   valid -  0.65926


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.69662   |   valid -  0.75556


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


...

train -  0.70074   |   valid -  0.62222


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.69605   |   valid -  0.61940


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.71087   |   valid -  0.63433
Average accuracy on crossval is 0.65871
Std is 0.04359
CPU times: user 1.03 s, sys: 12.6 ms, total: 1.04 s
Wall time: 1.05 s


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [39]:
svm_c01 = SVC(probability=True, kernel='linear', random_state=21, C=0.1)
crossval(svm_c01, X_train, y_train)


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.58120   |   valid -  0.55556


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.57543   |   valid -  0.56296


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


...

train -  0.59192   |   valid -  0.54815


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.59967   |   valid -  0.52985


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.57825   |   valid -  0.57463
Average accuracy on crossval is 0.56230
Std is 0.02177


In [ ]:
svm_c1 = SVC(probability=True, kernel='linear', random_state=21, C=1)
crossval(svm_c1, X_train, y_train)

/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.70486   |   valid -  0.65926


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.69662   |   valid -  0.75556


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


...

train -  0.70074   |   valid -  0.62222


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.69605   |   valid -  0.61940


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.71087   |   valid -  0.63433
Average accuracy on crossval is 0.65871
Std is 0.04359


In [41]:
svm_c10 = SVC(probability=True, kernel='linear', random_state=21, C=10)
crossval(svm_c10, X_train, y_train)


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.75021   |   valid -  0.72593


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.77741   |   valid -  0.82963


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


...

train -  0.77411   |   valid -  0.68889


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.78254   |   valid -  0.71642


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.78418   |   valid -  0.69403
Average accuracy on crossval is 0.72771
Std is 0.04417


## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [42]:
%%time
dt = DecisionTreeClassifier(max_depth=10, random_state=21)
crossval(dt, X_train, y_train)


train -  0.81039   |   valid -  0.74815
train -  0.77741   |   valid -  0.74074
train -  0.83347   |   valid -  0.70370
train -  0.79720   |   valid -  0.77037
train -  0.82440   |   valid -  0.75556
train -  0.80379   |   valid -  0.68889
train -  0.80709   |   valid -  0.76296
train -  0.80132   |   valid -  0.65926
train -  0.80807   |   valid -  0.74627
train -  0.80478   |   valid -  0.68657
Average accuracy on crossval is 0.72625
Std is 0.03635
CPU times: user 31.2 ms, sys: 1.33 ms, total: 32.5 ms
Wall time: 34.2 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [43]:
dt_5 = DecisionTreeClassifier(max_depth=5, random_state=21)
crossval(dt_5, X_train, y_train)


train -  0.59522   |   valid -  0.53333
train -  0.56307   |   valid -  0.53333
train -  0.60181   |   valid -  0.55556
train -  0.59604   |   valid -  0.57037
train -  0.60264   |   valid -  0.57778
train -  0.57955   |   valid -  0.53333
train -  0.58368   |   valid -  0.54815
train -  0.59275   |   valid -  0.51111
train -  0.58237   |   valid -  0.56716
train -  0.60132   |   valid -  0.50000
Average accuracy on crossval is 0.54301
Std is 0.02423


In [44]:
dt_15 = DecisionTreeClassifier(max_depth=15, random_state=21)
crossval(dt_15, X_train, y_train)


train -  0.95796   |   valid -  0.82963
train -  0.93075   |   valid -  0.85185
train -  0.95631   |   valid -  0.84444
train -  0.95301   |   valid -  0.86667
train -  0.95136   |   valid -  0.88148
train -  0.94724   |   valid -  0.83704
train -  0.95466   |   valid -  0.90370
train -  0.94806   |   valid -  0.83704
train -  0.95305   |   valid -  0.82090
train -  0.94316   |   valid -  0.85821
Average accuracy on crossval is 0.85310
Std is 0.02399


In [45]:
dt_20 = DecisionTreeClassifier(max_depth=20, random_state=21)
crossval(dt_20, X_train, y_train)


train -  0.98928   |   valid -  0.86667
train -  0.99011   |   valid -  0.89630
train -  0.98681   |   valid -  0.85185
train -  0.98763   |   valid -  0.90370
train -  0.98928   |   valid -  0.88148
train -  0.98186   |   valid -  0.86667
train -  0.98846   |   valid -  0.91852
train -  0.99093   |   valid -  0.89630
train -  0.99094   |   valid -  0.88060
train -  0.98847   |   valid -  0.88060
Average accuracy on crossval is 0.88427
Std is 0.01883


In [46]:
dt_leaf = DecisionTreeClassifier(max_depth=10, random_state=21, min_samples_leaf=5)
crossval(dt_leaf, X_train, y_train)


train -  0.75515   |   valid -  0.71111
train -  0.71476   |   valid -  0.66667
train -  0.78153   |   valid -  0.66667
train -  0.75268   |   valid -  0.73333
train -  0.76752   |   valid -  0.72593
train -  0.75268   |   valid -  0.67407
train -  0.74608   |   valid -  0.70370
train -  0.74279   |   valid -  0.61481
train -  0.75206   |   valid -  0.68657
train -  0.74465   |   valid -  0.64179
Average accuracy on crossval is 0.68247
Std is 0.03545


In [47]:
dt_split = DecisionTreeClassifier(max_depth=10, random_state=21, min_samples_split=10)
crossval(dt_split, X_train, y_train)


train -  0.79143   |   valid -  0.74074
train -  0.74691   |   valid -  0.71111
train -  0.81121   |   valid -  0.68148
train -  0.78236   |   valid -  0.74074
train -  0.80627   |   valid -  0.74815
train -  0.78236   |   valid -  0.67407
train -  0.78153   |   valid -  0.71852
train -  0.77988   |   valid -  0.62963
train -  0.78418   |   valid -  0.74627
train -  0.77842   |   valid -  0.67164
Average accuracy on crossval is 0.70624
Std is 0.03825


## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [48]:
%%time
rf = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)
crossval(rf, X_train, y_train)


train -  0.96373   |   valid -  0.87407
train -  0.97032   |   valid -  0.91111
train -  0.96867   |   valid -  0.88889
train -  0.97279   |   valid -  0.91111
train -  0.96785   |   valid -  0.91111
train -  0.96620   |   valid -  0.85185
train -  0.96867   |   valid -  0.91111
train -  0.96702   |   valid -  0.85185
train -  0.97199   |   valid -  0.88060
train -  0.96458   |   valid -  0.85075
Average accuracy on crossval is 0.88425
Std is 0.02499
CPU times: user 334 ms, sys: 4.76 ms, total: 339 ms
Wall time: 340 ms


### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [49]:
rf_100 = RandomForestClassifier(n_estimators=100, max_depth=14, random_state=21)
crossval(rf_100, X_train, y_train)


train -  0.96125   |   valid -  0.85926
train -  0.96702   |   valid -  0.91111
train -  0.97115   |   valid -  0.87407
train -  0.97197   |   valid -  0.88889
train -  0.97444   |   valid -  0.90370
train -  0.97115   |   valid -  0.85185
train -  0.96950   |   valid -  0.90370
train -  0.96538   |   valid -  0.83704
train -  0.97611   |   valid -  0.90299
train -  0.97199   |   valid -  0.85821
Average accuracy on crossval is 0.87908
Std is 0.02504


In [50]:
rf_d10 = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=21)
crossval(rf_d10, X_train, y_train)


train -  0.86150   |   valid -  0.77037
train -  0.86810   |   valid -  0.83704
train -  0.86727   |   valid -  0.78519
train -  0.88788   |   valid -  0.83704
train -  0.88788   |   valid -  0.84444
train -  0.86727   |   valid -  0.76296
train -  0.88871   |   valid -  0.85185
train -  0.88541   |   valid -  0.77037
train -  0.88138   |   valid -  0.79104
train -  0.87891   |   valid -  0.79104
Average accuracy on crossval is 0.80413
Std is 0.03278


In [51]:
rf_d20 = RandomForestClassifier(n_estimators=50, max_depth=20, random_state=21)
crossval(rf_d20, X_train, y_train)


train -  0.99753   |   valid -  0.88889
train -  0.99588   |   valid -  0.94815
train -  0.99505   |   valid -  0.88148
train -  0.99588   |   valid -  0.92593
train -  0.99753   |   valid -  0.91111
train -  0.99588   |   valid -  0.88148
train -  0.99505   |   valid -  0.91852
train -  0.99835   |   valid -  0.91111
train -  0.99918   |   valid -  0.91791
train -  0.99588   |   valid -  0.88060
Average accuracy on crossval is 0.90652
Std is 0.02159


In [52]:
rf_dnone = RandomForestClassifier(n_estimators=50, max_depth=None, random_state=21)
crossval(rf_dnone, X_train, y_train)


train -  1.00000   |   valid -  0.89630
train -  1.00000   |   valid -  0.94815
train -  1.00000   |   valid -  0.91111
train -  1.00000   |   valid -  0.93333
train -  1.00000   |   valid -  0.91111
train -  1.00000   |   valid -  0.90370
train -  1.00000   |   valid -  0.92593
train -  1.00000   |   valid -  0.91111
train -  1.00000   |   valid -  0.92537
train -  0.99918   |   valid -  0.87313
Average accuracy on crossval is 0.91392
Std is 0.01981


In [53]:
rf_leaf = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21, min_samples_leaf=5)
crossval(rf_leaf, X_train, y_train)


train -  0.76010   |   valid -  0.73333
train -  0.77906   |   valid -  0.72593
train -  0.75103   |   valid -  0.69630
train -  0.76505   |   valid -  0.71852
train -  0.76422   |   valid -  0.74815
train -  0.76917   |   valid -  0.71852
train -  0.77659   |   valid -  0.78519
train -  0.77411   |   valid -  0.63704
train -  0.77595   |   valid -  0.69403
train -  0.78501   |   valid -  0.71642
Average accuracy on crossval is 0.71734
Std is 0.03650


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [54]:
best_model = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {test_acc:.5f}')


Test accuracy: 0.89349


In [55]:
cm = confusion_matrix(y_test, y_pred)
class_counts = y_test.value_counts().sort_index()

for i in range(len(class_counts)):
    total = class_counts.iloc[i]
    errors = cm[i].sum() - cm[i, i]
    pct = (errors / total) * 100
    print(f'Class {i}: {errors} errors out of {total} ({pct:.2f}%)')


Class 0: 8 errors out of 27 (29.63%)
Class 1: 8 errors out of 55 (14.55%)
Class 2: 3 errors out of 30 (10.00%)
Class 3: 3 errors out of 80 (3.75%)
Class 4: 3 errors out of 21 (14.29%)
Class 5: 4 errors out of 54 (7.41%)
Class 6: 7 errors out of 71 (9.86%)


In [56]:
cm = confusion_matrix(y_test, y_pred)
class_counts = y_test.value_counts().sort_index()
worst_class = None
worst_pct = 0
for i in range(len(class_counts)):
    total = class_counts.iloc[i]
    errors = cm[i].sum() - cm[i, i]
    pct = (errors / total) * 100
    print(f'Class {i} ({class_counts.index[i]}): {errors}/{total} = {pct:.2f}%')
    if pct > worst_pct:
        worst_pct = pct
        worst_class = i
print(f'\nMost errors for class {worst_class}: {worst_pct:.2f}%')


Class 0 (0): 8/27 = 29.63%
Class 1 (1): 8/55 = 14.55%
Class 2 (2): 3/30 = 10.00%
Class 3 (3): 3/80 = 3.75%
Class 4 (4): 3/21 = 14.29%
Class 5 (5): 4/54 = 7.41%
Class 6 (6): 7/71 = 9.86%

Most errors for class 0: 29.63%


In [57]:
joblib.dump(best_model, 'best_model.joblib')


['best_model.joblib']

In [58]:
loaded = joblib.load('best_model.joblib')
print(f'Loaded model accuracy: {accuracy_score(y_test, loaded.predict(X_test)):.5f}')


Loaded model accuracy: 0.89349
